# GDTW

### 2.1 Loading Prepared Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import pickle
import gc
import warnings
import time
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import os

warnings.filterwarnings('ignore')

with open('data/ts_prepared_data.pkl', 'rb') as f:
    ts_data = pickle.load(f)

clustering_patients = ts_data['clustering_patients']
clustering_series = ts_data['clustering_series']
clustering_dates = ts_data['clustering_dates']
clustering_kzs_count = ts_data['clustering_kzs_count']
patient_groups = ts_data['patient_groups']

print(f"   Loaded patients for clustering: {len(clustering_patients)}")
print(f"   Available indicators: {list(clustering_series.keys())}")

# Check series lengths
lengths = [len(clustering_series['САД'][p]) for p in clustering_patients]
print(f"   Series length: min={min(lengths)}, max={max(lengths)}, mean={np.mean(lengths):.0f}")

### 2.2 GDTW Implementation

In [ ]:
def gt_dtw_distance(series1, series2, dates1, dates2, 
                    window_days=7, gamma=1.0, metric='САД'):
    n, m = len(series1), len(series2)
    
    # Convert datetime64 to days from start
    # Use minimum date as reference point
    min_date = np.min([np.min(dates1), np.min(dates2)])
    
    # Calculate days as (date - min_date) in days
    t1 = np.array([(d - min_date) / np.timedelta64(1, 'D') for d in dates1])
    t2 = np.array([(d - min_date) / np.timedelta64(1, 'D') for d in dates2])
    
    # Initialize DTW matrix
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0
    
    # Fill matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # Check time window
            time_diff = abs(t1[i-1] - t2[j-1])
            if time_diff > window_days:
                continue
            
            # Cost: value difference + temporal penalty
            value_cost = (series1[i-1] - series2[j-1]) ** 2
            time_penalty = gamma * (time_diff / window_days) ** 2
            
            cost = value_cost + time_penalty
            
            # Minimum cost from three transitions
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],    # insertion
                dtw_matrix[i, j-1],    # deletion
                dtw_matrix[i-1, j-1]   # match
            )
    
    return np.sqrt(dtw_matrix[n, m])

def gt_dtw_distance_fast(series1, series2, dates1, dates2, 
                         window_days=7, gamma=1.0, metric='САД'):
    """
    Fast GDTW version with window constraint (Sakoe-Chiba band)
    """
    n, m = len(series1), len(series2)
    
    # Convert datetime64 to days
    min_date = np.min([np.min(dates1), np.min(dates2)])
    t1 = np.array([(d - min_date) / np.timedelta64(1, 'D') for d in dates1])
    t2 = np.array([(d - min_date) / np.timedelta64(1, 'D') for d in dates2])
    
    # Limit window proportionally to series length
    window_size = max(1, int(window_days / 7))  # approximate window in steps
    
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0
    
    for i in range(1, n + 1):
        # Limit j by window
        j_start = max(1, i - window_size)
        j_end = min(m, i + window_size)
        
        for j in range(j_start, j_end + 1):
            time_diff = abs(t1[i-1] - t2[j-1])
            if time_diff > window_days:
                continue
            
            value_cost = (series1[i-1] - series2[j-1]) ** 2
            time_penalty = gamma * (time_diff / window_days) ** 2
            
            cost = value_cost + time_penalty
            
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],
                dtw_matrix[i, j-1],
                dtw_matrix[i-1, j-1]
            )
    
    return np.sqrt(dtw_matrix[n, m])

# Test on first two patients
print("\nTesting GDTW on first two patients:")
p1, p2 = clustering_patients[0], clustering_patients[1]

dist_sad = gt_dtw_distance(
    clustering_series['САД'][p1], clustering_series['САД'][p2],
    clustering_dates[p1], clustering_dates[p2],
    window_days=7, gamma=1.0
)
print(f"  GDTW distance (SBP): {dist_sad:.3f}")

# Test with different parameters
print("\nParameter influence:")
for window in [3, 7, 14]:
    dist = gt_dtw_distance(
        clustering_series['САД'][p1], clustering_series['САД'][p2],
        clustering_dates[p1], clustering_dates[p2],
        window_days=window, gamma=1.0
    )
    print(f"  window={window}: {dist:.3f}")

for g in [0.0, 0.5, 1.0, 2.0]:
    dist = gt_dtw_distance(
        clustering_series['САД'][p1], clustering_series['САД'][p2],
        clustering_dates[p1], clustering_dates[p2],
        window_days=7, gamma=g
    )
    print(f"  gamma={g}: {dist:.3f}")

### 2.3 Selecting Primary Indicator and Parameters

In [ ]:
# Check correlation between different indicators
sample_size = min(50, len(clustering_patients))
sample_patients = clustering_patients[:sample_size]

correlations = {'SBP-DBP': [], 'SBP-HR': [], 'DBP-HR': []}

for i, p1 in enumerate(sample_patients):
    for p2 in sample_patients[i+1:]:
        if p1 in clustering_series['САД'] and p2 in clustering_series['САД']:
            # Distances by different indicators
            d_sad = gt_dtw_distance(
                clustering_series['САД'][p1], clustering_series['САД'][p2],
                clustering_dates[p1], clustering_dates[p2],
                window_days=7, gamma=1.0
            )
            d_dad = gt_dtw_distance(
                clustering_series['ДАД'][p1], clustering_series['ДАД'][p2],
                clustering_dates[p1], clustering_dates[p2],
                window_days=7, gamma=1.0
            )
            d_chp = gt_dtw_distance(
                clustering_series['ЧП'][p1], clustering_series['ЧП'][p2],
                clustering_dates[p1], clustering_dates[p2],
                window_days=7, gamma=1.0
            )
            
            correlations['SBP-DBP'].append((d_sad, d_dad))
            correlations['SBP-HR'].append((d_sad, d_chp))
            correlations['DBP-HR'].append((d_dad, d_chp))

# Compute correlations
for key, pairs in correlations.items():
    if len(pairs) > 1:
        x, y = zip(*pairs)
        corr = np.corrcoef(x, y)[0, 1]
        print(f"  Distance correlation {key}: {corr:.3f}")

# Select SBP as primary indicator (most clinically significant)
primary_metric = 'САД'
print(f"\nSelected primary indicator: {primary_metric}")

### 2.4 Parallel Distance Matrix Computation

In [ ]:
import numpy as np
import time
from itertools import combinations
import pandas as pd

# Try importing numba (optional)
try:
    from numba import jit, prange
    HAS_NUMBA = True
except ImportError:
    HAS_NUMBA = False
    print("  numba not installed, using regular version")

def prepare_time_data(dates_dict):
    """
    Prepare timestamps: convert timedelta64 to days
    """
    # Find minimum date among all
    all_dates = []
    for d in dates_dict.values():
        if d is not None and len(d) > 0:
            all_dates.extend(d)
    
    if not all_dates:
        return {}
    
    # Convert all dates to pandas Timestamp for convenience
    min_date = pd.Timestamp(min(all_dates))
    
    # Create dictionary with days from start
    time_days = {}
    for patient, dates in dates_dict.items():
        if dates is not None and len(dates) > 0:
            # Convert to pandas Timestamp if needed
            if isinstance(dates[0], np.datetime64):
                dates_ts = [pd.Timestamp(d) for d in dates]
            else:
                dates_ts = dates
            
            # Calculate days from start
            days = np.array([(d - min_date).days for d in dates_ts], dtype=np.float32)
            time_days[patient] = days
    
    return time_days, min_date


@jit(nopython=True, cache=True)
def gt_dtw_distance_numba(s1, s2, t1, t2, window_days, gamma):
    """
    GDTW with JIT compilation - preserves real time relationship
    Optimization: pre-calculation of time differences
    """
    n, m = len(s1), len(s2)
    
    # Pre-compute time difference matrix
    # This avoids repeated computations in the loop
    time_diffs = np.zeros((n, m))
    for i in range(n):
        for j in range(m):
            time_diffs[i, j] = abs(t1[i] - t2[j])
    
    # Initialize DTW matrix
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0
    
    # Fill matrix row by row
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # Check time window (real time!)
            if time_diffs[i-1, j-1] > window_days:
                continue
            
            # Combined cost
            value_cost = (s1[i-1] - s2[j-1]) ** 2
            time_penalty = gamma * (time_diffs[i-1, j-1] / window_days) ** 2
            cost = value_cost + time_penalty
            
            # Minimum cost from three neighbors
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],
                dtw_matrix[i, j-1],
                dtw_matrix[i-1, j-1]
            )
    
    # If path not found, return penalty based on maximum difference
    if np.isinf(dtw_matrix[n, m]):
        # Estimate maximum possible distance
        max_val_diff = np.max(np.abs(s1 - np.mean(s1))) + np.max(np.abs(s2 - np.mean(s2)))
        return max_val_diff * np.sqrt(max(n, m))
    
    return np.sqrt(dtw_matrix[n, m])


@jit(nopython=True, parallel=True)
def gt_dtw_distance_numba_parallel(s1, s2, t1, t2, window_days, gamma):
    """
    Parallel version for even greater speedup
    """
    n, m = len(s1), len(s2)
    
    # Pre-compute time differences
    time_diffs = np.zeros((n, m))
    for i in range(n):
        for j in range(m):
            time_diffs[i, j] = abs(t1[i] - t2[j])
    
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0
    
    # Fill matrix with potential row parallelization
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if time_diffs[i-1, j-1] > window_days:
                continue
            
            value_cost = (s1[i-1] - s2[j-1]) ** 2
            time_penalty = gamma * (time_diffs[i-1, j-1] / window_days) ** 2
            cost = value_cost + time_penalty
            
            # Minimum from three neighbors
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],
                dtw_matrix[i, j-1],
                dtw_matrix[i-1, j-1]
            )
    
    if np.isinf(dtw_matrix[n, m]):
        max_val_diff = np.max(np.abs(s1 - np.mean(s1))) + np.max(np.abs(s2 - np.mean(s2)))
        return max_val_diff * np.sqrt(max(n, m))
    
    return np.sqrt(dtw_matrix[n, m])


def gt_dtw_distance_fast(s1, s2, t1, t2, window_days, gamma):
    """
    Fast version without numba (with pre-computed time differences)
    """
    n, m = len(s1), len(s2)
    
    # Vectorized time difference calculation
    t1_expanded = t1.reshape(-1, 1)
    t2_expanded = t2.reshape(1, -1)
    time_diffs = np.abs(t1_expanded - t2_expanded)
    
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0
    
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if time_diffs[i-1, j-1] > window_days:
                continue
            
            value_cost = (s1[i-1] - s2[j-1]) ** 2
            time_penalty = gamma * (time_diffs[i-1, j-1] / window_days) ** 2
            cost = value_cost + time_penalty
            
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],
                dtw_matrix[i, j-1],
                dtw_matrix[i-1, j-1]
            )
    
    if np.isinf(dtw_matrix[n, m]):
        max_val_diff = np.max(np.abs(s1 - np.mean(s1))) + np.max(np.abs(s2 - np.mean(s2)))
        return max_val_diff * np.sqrt(max(n, m))
    
    return np.sqrt(dtw_matrix[n, m])


def compute_distance_matrix_optimized(patients, series_dict, dates_dict, 
                                      metric='САД', window_days=7, gamma=1.0,
                                      use_parallel=False):
    """
    OPTIMIZED version preserving temporal relationship
    """
    n = len(patients)
    
    print("  Preparing timestamps...")
    time_days, min_date = prepare_time_data(dates_dict)
    print(f"  Minimum date: {min_date}")
    
    print("  Preparing patient data...")
    all_series = []
    all_times = []
    valid_patients = []
    
    for p in patients:
        s = series_dict[metric].get(p)
        t = time_days.get(p)
        
        if s is not None and t is not None and len(s) > 0 and len(t) > 0:
            all_series.append(s.astype(np.float32))
            all_times.append(t)
            valid_patients.append(p)
    
    n_valid = len(valid_patients)
    print(f"  Valid patients: {n_valid}/{n}")
    
    if n_valid == 0:
        return np.array([]), []
    
    # Choose distance function
    if HAS_NUMBA:
        if use_parallel:
            dist_func = gt_dtw_distance_numba_parallel
            print("  Using parallel JIT version")
        else:
            dist_func = gt_dtw_distance_numba
            print("  Using JIT-compiled version")
    else:
        dist_func = gt_dtw_distance_fast
        print("  Using optimized version without numba")
    
    # Initialize matrix
    distance_matrix = np.zeros((n_valid, n_valid), dtype=np.float32)
    
    total_pairs = n_valid * (n_valid - 1) // 2
    print(f"  Total pairs to compute: {total_pairs:,}")
    
    # Compute distances
    start_time = time.time()
    processed = 0
    
    # For tracking time per pair
    pair_times = []
    
    for i, j in combinations(range(n_valid), 2):
        pair_start = time.time()
        
        dist = dist_func(
            all_series[i], all_series[j],
            all_times[i], all_times[j],
            window_days, gamma
        )
        
        pair_time = time.time() - pair_start
        pair_times.append(pair_time)
        
        distance_matrix[i, j] = dist
        distance_matrix[j, i] = dist
        
        processed += 1
        
        # Progress every 5%
        if processed % max(1, total_pairs // 20) == 0:
            elapsed = time.time() - start_time
            pct = processed / total_pairs * 100
            avg_pair_time = np.mean(pair_times[-100:])  # average of last 100 pairs
            remaining_pairs = total_pairs - processed
            remaining_time = remaining_pairs * avg_pair_time
            
            print(f"    Progress: {pct:.1f}% ({processed}/{total_pairs}), "
                  f"elapsed: {elapsed:.1f}s, remaining: {remaining_time:.1f}s "
                  f"(avg per pair: {avg_pair_time*1000:.2f}ms)")
    
    total_time = time.time() - start_time
    print(f"  Matrix computed in {total_time:.1f} sec")
    
    if processed > 0:
        avg_time = total_time / processed
        print(f"  Average time per pair: {avg_time*1000:.2f} ms")
    
    return distance_matrix, valid_patients


def compute_distance_matrix_sampled(patients, series_dict, dates_dict,
                                    metric='САД', window_days=7, gamma=1.0,
                                    sample_size=1500, use_parallel=False):
    """
    Version with smart sampling for very large datasets
    """
    if len(patients) <= sample_size:
        return compute_distance_matrix_optimized(
            patients, series_dict, dates_dict,
            metric, window_days, gamma, use_parallel
        )
    
    print(f"\n  Sampling {sample_size} out of {len(patients)} patients")
    
    # Stratified sampling by series length
    lengths = []
    valid_patients_with_length = []
    
    for p in patients:
        s = series_dict[metric].get(p)
        if s is not None:
            lengths.append(len(s))
            valid_patients_with_length.append(p)
    
    # Sort by length and take uniformly
    sorted_idx = np.argsort(lengths)
    step = len(sorted_idx) // sample_size
    sampled_idx = sorted_idx[::step][:sample_size]
    sampled_patients = [valid_patients_with_length[i] for i in sampled_idx]
    
    print(f"  Length range in sample: {min(lengths[i] for i in sampled_idx)} - "
          f"{max(lengths[i] for i in sampled_idx)} days")
    
    return compute_distance_matrix_optimized(
        sampled_patients, series_dict, dates_dict,
        metric, window_days, gamma, use_parallel
    )

print(f"\nComputing distance matrix for {len(clustering_patients)} patients...")

# Settings
WINDOW_DAYS = 14  # Increase window for better coverage
GAMMA = 1.0
USE_PARALLEL = True  # Enable parallel version if needed

# Choose strategy
n_patients = len(clustering_patients)

if n_patients > 8000:
    print(f"  Many patients ({n_patients}), using sampling")
    distance_matrix, valid_patients = compute_distance_matrix_sampled(
        clustering_patients,
        clustering_series,
        clustering_dates,
        metric='САД',
        window_days=WINDOW_DAYS,
        gamma=GAMMA,
        sample_size=7389,
        use_parallel=USE_PARALLEL
    )
else:
    print(f"  Using full version")
    distance_matrix, valid_patients = compute_distance_matrix_optimized(
        clustering_patients,
        clustering_series,
        clustering_dates,
        metric='САД',
        window_days=WINDOW_DAYS,
        gamma=GAMMA,
        use_parallel=USE_PARALLEL
    )

# Update patient list
clustering_patients = valid_patients

# Check matrix
if len(distance_matrix) > 0:
    print(f"\nDistance matrix check:")
    print(f"  Shape: {distance_matrix.shape}")
    print(f"  Data type: {distance_matrix.dtype}")
    print(f"  Memory used: {distance_matrix.nbytes / 1024**2:.1f} MB")
    
    # Statistics (upper triangle only)
    triu_indices = np.triu_indices_from(distance_matrix, k=1)
    distances = distance_matrix[triu_indices]
    
    if len(distances) > 0:
        # Filter inf and very large values
        valid_distances = distances[np.isfinite(distances)]
        if len(valid_distances) > 0:
            print(f"  Minimum distance: {np.min(valid_distances):.3f}")
            print(f"  Maximum distance: {np.max(valid_distances):.3f}")
            print(f"  Mean distance: {np.mean(valid_distances):.3f}")
            print(f"  Median distance: {np.median(valid_distances):.3f}")
            print(f"  Number of inf: {np.sum(~np.isfinite(distances))}")
            print(f"  Matrix density: {len(valid_distances) / distances.size * 100:.2f}%")
    
    # Save matrix
    np.save('data/gt_dtw_distance_matrix.npy', distance_matrix)
    print(f"\nSaved: data/gt_dtw_distance_matrix.npy")
    
    # Save updated patient list
    import pickle
    with open('data/clustering_patients_updated.pkl', 'wb') as f:
        pickle.dump(clustering_patients, f)
    print(f"Saved: data/clustering_patients_updated.pkl")
else:
    print("  Error: distance matrix is empty!")

### 2.5 Distance Matrix Visualization

In [ ]:
# 2.5 Distance Matrix Visualization for sparse matrix
distance_matrix = np.load('data/gt_dtw_distance_matrix.npy')
print(f"\nMatrix density: {np.sum((distance_matrix > 0) & np.isfinite(distance_matrix)) / distance_matrix.size * 100:.4f}%")
print(f"Pairs computed: {np.sum((distance_matrix > 0) & np.isfinite(distance_matrix)) // 2:.0f} out of {2500*2499//2}")

# Sort patients by CSE count
sorted_indices = np.argsort([clustering_kzs_count.get(p, 0) for p in clustering_patients])
sorted_matrix = distance_matrix[sorted_indices][:, sorted_indices]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Heatmap (ONLY with finite non-zero values)
# Create masked matrix (mask zeros and infinities)
masked_matrix = np.ma.masked_where((sorted_matrix == 0) | (~np.isfinite(sorted_matrix)), sorted_matrix)

# Determine range for color scale (only finite values)
valid_values = sorted_matrix[(sorted_matrix > 0) & np.isfinite(sorted_matrix)]
if len(valid_values) > 0:
    vmin = float(np.min(valid_values))
    vmax = float(np.max(valid_values))
    print(f"Distance range: {vmin:.2f} - {vmax:.2f}")
else:
    vmin, vmax = 0, 30
    print("No valid values, using default range")

im1 = axes[0].imshow(masked_matrix, cmap='plasma', aspect='auto', 
                     interpolation='none', vmin=vmin, vmax=vmax)
axes[0].set_title(f'GDTW Distance Matrix\n(showing only {len(valid_values):,} finite values)')
axes[0].set_xlabel('Patients (sorted by CSE)')
axes[0].set_ylabel('Patients (sorted by CSE)')
plt.colorbar(im1, ax=axes[0], label='GDTW distance')

# Add sparsity information
axes[0].text(0.02, 0.98, f'Density: 0.{np.sum((distance_matrix > 0) & np.isfinite(distance_matrix)) / distance_matrix.size * 100:.4f}%', 
             transform=axes[0].transAxes, fontsize=10,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 2. Distance distribution (only finite values)
valid_distances = distance_matrix[(distance_matrix > 0) & np.isfinite(distance_matrix)]
if len(valid_distances) > 0:
    axes[1].hist(valid_distances.flatten(), bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[1].axvline(np.median(valid_distances), color='red', linestyle='--', linewidth=2,
                    label=f'Median: {np.median(valid_distances):.2f}')
    axes[1].axvline(np.mean(valid_distances), color='green', linestyle='--', linewidth=2,
                    label=f'Mean: {np.mean(valid_distances):.2f}')
    axes[1].set_xlabel('GDTW distance')
    axes[1].set_ylabel('Number of pairs')
    axes[1].set_title(f'Distance Distribution\n(n={len(valid_distances):,} pairs)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Add min/max information
    axes[1].text(0.05, 0.95, f'Min: {np.min(valid_distances):.2f}\nMax: {np.max(valid_distances):.2f}', 
                 transform=axes[1].transAxes, fontsize=9,
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 3. Relationship with CSE difference
# Collect data only for computed pairs (finite values)
distances_list = []
kzs_diffs_list = []

# Take only upper triangle
for i in range(len(clustering_patients)):
    for j in range(i+1, len(clustering_patients)):
        if distance_matrix[i, j] > 0 and np.isfinite(distance_matrix[i, j]):
            distances_list.append(distance_matrix[i, j])
            kzs_i = clustering_kzs_count.get(clustering_patients[i], 0)
            kzs_j = clustering_kzs_count.get(clustering_patients[j], 0)
            kzs_diffs_list.append(abs(kzs_i - kzs_j))

if len(distances_list) > 0:
    # Convert to arrays
    distances_arr = np.array(distances_list)
    kzs_diffs_arr = np.array(kzs_diffs_list)
    
    # Sample if too many points
    if len(distances_arr) > 5000:
        sample_idx = np.random.choice(len(distances_arr), 5000, replace=False)
        distances_arr = distances_arr[sample_idx]
        kzs_diffs_arr = kzs_diffs_arr[sample_idx]
    
    scatter = axes[2].scatter(kzs_diffs_arr, distances_arr, 
                              alpha=0.5, s=10, c=kzs_diffs_arr, cmap='viridis')
    axes[2].set_xlabel('CSE count difference')
    axes[2].set_ylabel('GDTW distance')
    axes[2].set_title(f'Relationship with CSE Difference\n(n={len(distances_list)} pairs)')
    plt.colorbar(scatter, ax=axes[2], label='CSE difference')
    
    # Trend line
    if len(distances_arr) > 1:
        z = np.polyfit(kzs_diffs_arr, distances_arr, 1)
        x_line = np.linspace(min(kzs_diffs_arr), max(kzs_diffs_arr), 100)
        axes[2].plot(x_line, np.poly1d(z)(x_line), "r--", linewidth=2,
                    label=f'Trend: y={z[0]:.3f}x+{z[1]:.2f}')
        axes[2].legend()
    
    # Add correlation information
    corr = np.corrcoef(kzs_diffs_arr, distances_arr)[0, 1]
    axes[2].text(0.05, 0.95, f'Correlation: {corr:.3f}', 
                 transform=axes[2].transAxes, fontsize=10,
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Add CSE difference statistics
    axes[2].text(0.05, 0.85, f'ΔCSE: {int(min(kzs_diffs_arr))}-{int(max(kzs_diffs_arr))}', 
                 transform=axes[2].transAxes, fontsize=9,
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
else:
    axes[2].text(0.5, 0.5, 'No data for analysis', ha='center', va='center')
    axes[2].set_title('Distance vs CSE Difference Relationship')

plt.tight_layout()
plt.savefig('data/photo/gt_dtw_distance_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSaved: data/photo/gt_dtw_distance_matrix.png")
print(f"Visualization statistics:")
print(f"  - Non-zero values displayed: {len(valid_distances):,}")
if len(valid_values) > 0:
    print(f"  - Distance range: {vmin:.2f} - {vmax:.2f}")

### 2.6 Agglomerative Clustering

In [ ]:
# 2.6 Agglomerative Clustering
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform

distance_matrix = np.load('data/gt_dtw_distance_matrix.npy')
condensed_dist = squareform(distance_matrix)

# Try different linkage methods
print("Computing linkage matrices...")
methods = ['ward', 'complete', 'average']
linkage_matrices = {}

for method in methods:
    print(f"  Method {method}...")
    linkage_matrices[method] = linkage(condensed_dist, method=method)

# Visualize dendrograms
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (method, link_mat) in zip(axes, linkage_matrices.items()):
    # Truncated dendrogram for clarity
    dendrogram(link_mat, ax=ax, truncate_mode='level', p=6,
               show_leaf_counts=True, leaf_rotation=90.)
    ax.set_title(f'Dendrogram (method: {method})')
    ax.set_xlabel('Patients')
    ax.set_ylabel('Distance')

plt.tight_layout()
plt.savefig('data/photo/dendrograms_comparison.png', dpi=150)
plt.show()
print("Saved: data/photo/dendrograms_comparison.png")

### 2.7 Determining Optimal Number of Clusters

In [ ]:
# 2.7 Determining Optimal Number of Clusters
from sklearn.metrics import silhouette_score

# Use Ward method (usually best)
linkage_matrix = linkage_matrices['ward']

# Silhouette score analysis for different cluster numbers
k_range = range(2, min(16, len(clustering_patients) // 10))
silhouette_scores = []
inertia = []

print("Analyzing silhouette scores...")
for k in k_range:
    labels = fcluster(linkage_matrix, k, criterion='maxclust')
    score = silhouette_score(distance_matrix, labels, metric='precomputed')
    silhouette_scores.append(score)
    
    # Compute "inertia" (sum of squares within clusters)
    cluster_inertia = 0
    for cluster_id in range(1, k + 1):
        cluster_points = np.where(labels == cluster_id)[0]
        if len(cluster_points) > 1:
            cluster_distances = distance_matrix[np.ix_(cluster_points, cluster_points)]
            cluster_inertia += np.sum(cluster_distances) / 2
    inertia.append(cluster_inertia)
    
    print(f"  k={k}: silhouette={score:.3f}")

# Find optimal k
optimal_k_silhouette = k_range[np.argmax(silhouette_scores)]
print(f"\nOptimal k by silhouette: {optimal_k_silhouette}")

# Elbow method
k_derivative = np.diff(inertia)
k_elbow = k_range[np.argmin(k_derivative) + 1] if len(k_derivative) > 0 else 3
print(f"Optimal k by elbow method: {k_elbow}")

# Choose k
optimal_k = min(optimal_k_silhouette, k_elbow)  # or can take average
optimal_k = 3
print(f"Selected number of clusters: {optimal_k}")
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Silhouette scores
axes[0].plot(k_range, silhouette_scores, 'bo-')
axes[0].axvline(x=optimal_k_silhouette, color='red', linestyle='--', 
                label=f'Optimum: {optimal_k_silhouette}')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Silhouette Score')
axes[0].set_title('Silhouette Analysis')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Elbow method
axes[1].plot(k_range, inertia, 'ro-')
axes[1].axvline(x=k_elbow, color='blue', linestyle='--', 
                label=f'Elbow: {k_elbow}')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Inertia (sum of intra-cluster distances)')
axes[1].set_title('Elbow Method')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/photo/optimal_k_analysis.png', dpi=150)
plt.show()
print("Saved: data/photo/optimal_k_analysis.png")

### 2.8 Obtaining Final Clusters

In [ ]:
# 2.8 Obtaining Final Clusters
cluster_labels = fcluster(linkage_matrix, optimal_k, criterion='maxclust')

# Create dataframe with cluster information
cluster_df = pd.DataFrame({
    'patient_id': clustering_patients,
    'cluster': cluster_labels,
    'kzs_count': [clustering_kzs_count.get(p, 0) for p in clustering_patients],
    'group': [patient_groups.get(p, 'unknown') for p in clustering_patients]
})

# Cluster statistics
print(f"\nCluster distribution:")
cluster_stats = cluster_df.groupby('cluster').agg({
    'patient_id': 'count',
    'kzs_count': ['mean', 'median', 'std']
}).round(1)
cluster_stats.columns = ['size', 'kzs_mean', 'kzs_median', 'kzs_std']
print(cluster_stats)

print(f"\nObservation group distribution by cluster:")
group_by_cluster = pd.crosstab(
    cluster_df['cluster'], 
    cluster_df['group'],
    normalize='index'
) * 100
print(group_by_cluster.round(1))

# Save clusters
cluster_df.to_csv('data/clustering_results.csv', index=False)
print(f"\nSaved: data/clustering_results.csv")

### 2.9 Cluster Visualization

In [ ]:
# 2.9 Cluster Visualization
# Function to compute DTW centroid
def dtw_barycenter_averaging(series_list, max_iter=10):
    """Simplified DBA version"""
    if not series_list:
        return None
    
    # Use the longest series as initial approximation
    longest_idx = np.argmax([len(s) for s in series_list])
    centroid = series_list[longest_idx].copy()
    
    # Several averaging iterations
    for _ in range(max_iter):
        aligned_sum = np.zeros_like(centroid, dtype=float)
        aligned_count = np.zeros_like(centroid, dtype=int)
        
        for series in series_list:
            # Simple alignment by length (for demonstration)
            if len(series) >= len(centroid):
                aligned = series[:len(centroid)]
            else:
                # Interpolation
                x_old = np.linspace(0, 1, len(series))
                x_new = np.linspace(0, 1, len(centroid))
                aligned = np.interp(x_new, x_old, series)
            
            aligned_sum += aligned
            aligned_count += 1
        
        new_centroid = aligned_sum / aligned_count
        if np.allclose(centroid, new_centroid):
            break
        centroid = new_centroid
    
    return centroid

# Visualize cluster centroids
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for cluster_id in range(1, optimal_k + 1):
    if cluster_id > len(axes):
        break
        
    ax = axes[cluster_id - 1]
    
    cluster_patients = cluster_df[cluster_df['cluster'] == cluster_id]['patient_id'].tolist()
    
    # Get series for these patients
    cluster_series = []
    for p in cluster_patients[:10]:  # limit for speed
        if p in clustering_series['САД']:
            cluster_series.append(clustering_series['САД'][p])
    
    if cluster_series:
        # Display several examples
        for series in cluster_series[:5]:
            days = np.arange(len(series))
            ax.plot(days, series, alpha=0.3, linewidth=0.5, color='gray')
        
        # Compute and display centroid
        centroid = dtw_barycenter_averaging(cluster_series)
        if centroid is not None:
            days = np.arange(len(centroid))
            ax.plot(days, centroid, linewidth=2, color='red', label='Centroid')
        
        # Statistics
        kzs_mean = cluster_df[cluster_df['cluster'] == cluster_id]['kzs_count'].mean()
        size = len(cluster_patients)
        ax.set_title(f'Cluster {cluster_id} (n={size}, CSE={kzs_mean:.1f})')
        ax.set_xlabel('Days')
        ax.set_ylabel('SBP (normalized)')
        ax.legend()
        ax.grid(True, alpha=0.3)

# Remove extra subplots
for idx in range(optimal_k, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig('data/photo/cluster_centroids.png', dpi=150)
plt.show()
print("Saved: data/photo/cluster_centroids.png")

### 2.10 Visualization of Patients Closest to Cluster Centroids

In [ ]:
# 2.10 Visualization of Patients Closest to Cluster Centroids
# (normalized time scale 0-100%)

def find_closest_patient_to_centroid(cluster_series_list, centroid):
    """
    Finds the patient series closest to the centroid
    """
    if centroid is None or not cluster_series_list:
        return None, None
    
    min_distance = float('inf')
    closest_series = None
    closest_idx = None
    
    for idx, series in enumerate(cluster_series_list):
        # Align to same length for comparison
        if len(series) >= len(centroid):
            aligned = series[:len(centroid)]
        else:
            x_old = np.linspace(0, 1, len(series))
            x_new = np.linspace(0, 1, len(centroid))
            aligned = np.interp(x_new, x_old, series)
        
        # Euclidean distance
        distance = np.sqrt(np.mean((aligned - centroid) ** 2))
        
        if distance < min_distance:
            min_distance = distance
            closest_series = series
            closest_idx = idx
    
    return closest_series, closest_idx

# Create 1x3 plot for three clusters
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Dictionary to store closest patient information
closest_patients_info = {}

for cluster_id in range(1, optimal_k + 1):
    ax = axes[cluster_id - 1]
    cluster_patients = cluster_df[cluster_df['cluster'] == cluster_id]['patient_id'].tolist()
    cluster_series = []
    valid_patients = []
    for p in cluster_patients:
        if p in clustering_series['САД']:
            cluster_series.append(clustering_series['САД'][p])
            valid_patients.append(p)
    
    if cluster_series:
        # Compute centroid
        centroid = dtw_barycenter_averaging(cluster_series)
        
        if centroid is not None:
            # Find patient closest to centroid
            closest_series, closest_idx = find_closest_patient_to_centroid(cluster_series, centroid)
            
            if closest_series is not None and closest_idx is not None:
                closest_patient_id = valid_patients[closest_idx]
                
                # Save information
                closest_patients_info[cluster_id] = {
                    'patient_id': closest_patient_id,
                    'series_length': len(closest_series),
                    'centroid_length': len(centroid)
                }
                
                # NORMALIZED TIME SCALE (0-100%)
                # For patient
                x_patient = np.linspace(0, 100, len(closest_series))
                ax.plot(x_patient, closest_series, alpha=0.8, linewidth=1.5, color='blue',
                       label=f'Patient {closest_patient_id[:8]}...')
                
                # For centroid
                x_centroid = np.linspace(0, 100, len(centroid))
                ax.plot(x_centroid, centroid, linewidth=2, color='red', label='Centroid')
                
                # Statistics
                kzs_mean = cluster_df[cluster_df['cluster'] == cluster_id]['kzs_count'].mean()
                size = len(cluster_patients)
                patient_kzs = cluster_df[cluster_df['patient_id'] == closest_patient_id]['kzs_count'].values[0]
                
                ax.set_title(f'Cluster {cluster_id} (n={size}, CSE mean={kzs_mean:.1f})\n'
                           f'Patient: {closest_patient_id} (CSE={patient_kzs})')
                ax.set_xlabel('Observation period percentage (%)')
                ax.set_ylabel('SBP (norm.)')
                ax.legend(loc='best')
                ax.grid(True, alpha=0.3)
                ax.set_xlim(0, 100)

plt.tight_layout()
plt.savefig('data/photo/cluster_representative_patients_normalized_time.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: data/photo/cluster_representative_patients_normalized_time.png")

# Print series length information
for cluster_id, info in closest_patients_info.items():
    print(f"Cluster {cluster_id}:")
    print(f"  Patient: {info['patient_id']}")
    print(f"  Patient series length: {info['series_length']} days")
    print(f"  Centroid length: {info['centroid_length']} days")
    print()

In [ ]:
# Original values for these patients for clinician reference
# Load original dataframe with measurements
df_raw = pd.read_csv('data/clean_primary_dedup.csv', parse_dates=['время измерения'])
df_raw['дата'] = pd.to_datetime(df_raw['время измерения'].dt.date)

# Create daily aggregates (as in main code)
daily_stats_raw = df_raw.groupby(['id пациента', 'дата']).agg({
    'САД': 'mean',
    'ДАД': 'mean',
    'ЧП': 'mean'
}).round(1).reset_index()

# Convert ID to string
daily_stats_raw['id_patient_str'] = daily_stats_raw['id пациента'].astype(str)

for cluster_id, info in closest_patients_info.items():
    pid = info['patient_id']
    patient_kzs = cluster_df[cluster_df['patient_id'] == pid]['kzs_count'].values[0]
    
    print(f"\nCluster {cluster_id}, patient {pid} (CSE={patient_kzs}):")
    print(f"  Observation length: {info['series_length']} days")
    
    # Get patient data
    patient_data = daily_stats_raw[daily_stats_raw['id_patient_str'] == pid]
    
    if len(patient_data) > 0:
        # SBP
        sad_vals = patient_data['САД'].values
        print(f"  SBP: mean={np.mean(sad_vals):.1f}, min={np.min(sad_vals):.1f}, max={np.max(sad_vals):.1f}, {len(sad_vals)} days")
        
        # DBP
        dad_vals = patient_data['ДАД'].values
        print(f"  DBP: mean={np.mean(dad_vals):.1f}, min={np.min(dad_vals):.1f}, max={np.max(dad_vals):.1f}, {len(dad_vals)} days")
        
        # HR
        chp_vals = patient_data['ЧП'].values
        print(f"  HR: mean={np.mean(chp_vals):.1f}, min={np.min(chp_vals):.1f}, max={np.max(chp_vals):.1f}, {len(chp_vals)} days")
    else:
        print("  Data not found")

In [ ]:
# Load anthropometric data from original file
df_raw = pd.read_csv('data/clean_primary_dedup.csv', parse_dates=['время измерения'])

# Take unique values per patient (to avoid day duplication)
demographics = df_raw.groupby('id пациента').agg({
    'рост': 'first',
    'масса': 'first',
    'возраст': 'first'
}).reset_index()

# Convert ID to string
demographics['id_patient_str'] = demographics['id пациента'].astype(str)

# Add cluster information
demographics['cluster'] = demographics['id_patient_str'].map(
    dict(zip(cluster_df['patient_id'], cluster_df['cluster']))
)

# Remove patients without cluster
demographics = demographics.dropna(subset=['cluster'])

# Print statistics by cluster
print("Anthropometrics by cluster:")

cluster_stats = demographics.groupby('cluster').agg({
    'масса': ['mean', 'std', 'count'],
    'рост': ['mean', 'std'],
    'возраст': ['mean', 'std']
}).round(1)

print(cluster_stats)

### 2.11 Saving All Results for Subsequent Stages

In [ ]:
# 2.11 Saving All Results for Subsequent Stages
# Save all results
results = {
    'clustering_patients': clustering_patients,
    'cluster_labels': cluster_labels,
    'cluster_df': cluster_df,
    'distance_matrix': distance_matrix,
    'linkage_matrix': linkage_matrix,
    'optimal_k': optimal_k,
    'silhouette_scores': silhouette_scores,
    'k_range': list(k_range),
    'params': {
        'window_days': 7,
        'gamma': 1.0,
        'metric': 'САД'
    }
}

with open('data/clustering_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print(f"Saved: data/clustering_results.pkl")
print(f"""
1. GDTW implemented:
   - Time window: {results['params']['window_days']} days
   - Parameter gamma = {results['params']['gamma']}
   - Primary indicator: {results['params']['metric']}
2. Distance matrix computed:
   - Patients: {len(clustering_patients)}
   - Matrix size: {distance_matrix.shape}
   - Distance range: {np.min(distance_matrix[distance_matrix > 0]):.3f} - {np.max(distance_matrix):.3f}
3. Agglomerative clustering performed:
   - Linkage method: ward
   - Optimal number of clusters: {optimal_k}
   - Silhouette score: {np.max(silhouette_scores):.3f}
4. Cluster characteristics:
""")

for cluster_id in range(1, optimal_k + 1):
    cluster_data = cluster_df[cluster_df['cluster'] == cluster_id]
    print(f"   Cluster {cluster_id}: {len(cluster_data)} patients, "
          f"mean CSE={cluster_data['kzs_count'].mean():.1f}")

# Memory cleanup
#del distance_matrix, linkage_matrix, clustering_series, clustering_dates
del cluster_df, results
gc.collect()